# Amazon ML Challenge 2026 - Phase 1 Exploratory Data Analysis

This notebook inspects the real challenge dataset structure, source distributions, ground truth characteristics, field-level noise patterns, and train/test leakage risks.

**Strict Training Scope:** All statistics and noise patterns are derived strictly from the training split (`train_source1.tsv`, `train_source2.tsv`, `train_source3.tsv`, and `train_ground_truth.tsv`). Test files are verified for presence and structure only.

In [ ]:
import sys
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

if hasattr(sys.stdout, 'reconfigure'):
    try:
        sys.stdout.reconfigure(encoding='utf-8', errors='replace')
    except Exception:
        pass

# Robust profile path resolution (works from project root or notebooks dir)
profile_path = Path("output/dataset_profile.json")
if not profile_path.exists():
    profile_path = Path("../output/dataset_profile.json")

with open(profile_path, "r", encoding="utf-8") as f:
    profile = json.load(f)

print(f"Loaded dataset profile generated at: {profile.get('generated_at')}")
print(f"Total training rows profiled: {profile['totals']['rows_profiled']:,}")

In [ ]:
# 1. Discovered Challenge Files
file_rows = []
for f in profile["files"]:
    cols = f.get("columns", [])
    col_names = [c["name"] if isinstance(c, dict) else str(c) for c in cols]
    file_rows.append({
        "Filename": f["filename"],
        "Split": f["split"],
        "Role": f["role"],
        "Size (MB)": f"{f.get('size_mib', 0):.2f}",
        "Rows": f"{f['row_count']:,}" if f.get("row_count") is not None else "Out of scope (unopened)",
        "Columns": ", ".join(col_names)
    })
df_files = pd.DataFrame(file_rows)
df_files

In [ ]:
# 2. Source ID Structure and Statistics
id_rows = []
for item in profile.get("identifiers", []):
    id_rows.append({
        "File": Path(item["file"]).name,
        "Source Prefix": item["prefix"],
        "Total Rows": f"{item['row_count']:,}",
        "Unique IDs": f"{item['distinct_count']:,}",
        "Duplicates": item["duplicate_id_count"],
        "Sample IDs": ", ".join(item["sample_values"][:3])
    })
df_ids = pd.DataFrame(id_rows)
df_ids

In [ ]:
# 3. Ground Truth Analysis
gt = profile["ground_truth"]
total_s1 = gt['total_rows']
zero_m = gt['s1_entities_with_zero_matches']
one_m = gt['s1_entities_with_one_match']
multi_m = gt['s1_entities_with_multiple_matches']

print("=== Training Ground Truth Summary ===")
print(f"Total S1 Entities in GT: {total_s1:,}")
print(f"S1 with 0 Matches (Singletons): {zero_m:,} ({zero_m / total_s1 * 100:.2f}%)")
print(f"S1 with 1 Match: {one_m:,} ({one_m / total_s1 * 100:.2f}%)")
print(f"S1 with Multiple Matches: {multi_m:,} ({multi_m / total_s1 * 100:.2f}%)")
print(f"Average Matches per S1: {gt['average_matches_per_s1']:.3f}")
print(f"Median Matches per S1: {gt['median_matches_per_s1']}")
print(f"Maximum Matches per S1: {gt['max_matches_per_s1']}")
print("\nMatch Contribution by Source:")
for prefix, count in gt["match_contribution_by_prefix"].items():
    print(f"  Source {prefix}: {count:,} matches ({count / gt['total_matched_ids'] * 100:.2f}%)")

In [ ]:
# 4. Ground Truth Match Cardinality Distribution
dist = gt["match_count_distribution"]
df_dist = pd.DataFrame([{"Matches": int(k), "Count": v} for k, v in dist.items()]).sort_values("Matches")

plt.figure(figsize=(10, 5))
plt.bar(df_dist["Matches"], df_dist["Count"], color="#232f3e", edgecolor="#ff9900", linewidth=1.5)
plt.xlabel("Number of Matched Entities per S1 Entity", fontsize=12)
plt.ylabel("Count of S1 Entities", fontsize=12)
plt.title("Ground Truth Match Cardinality Distribution (Train)", fontsize=14, fontweight="bold")
plt.xticks(df_dist["Matches"])
plt.grid(axis="y", linestyle="--", alpha=0.5)
for _, row in df_dist.iterrows():
    plt.text(row["Matches"], row["Count"] + 15000, f"{row['Count']:,}", ha="center", fontsize=8, rotation=45)
plt.ylim(0, df_dist["Count"].max() * 1.15)
plt.tight_layout()
plt.show()

In [ ]:
# 5. Field Missingness and Characteristics
field_stats = []
for f in profile["files"]:
    if f.get("split") != "train" or f.get("role") != "source":
        continue
    for col in f.get("columns", []):
        if not isinstance(col, dict):
            continue
        field_stats.append({
            "File": f["filename"],
            "Field": col["name"],
            "Inferred Dtype": col["inferred_dtype"],
            "Null Count": f"{col['null_count']:,}",
            "Null %": f"{col['null_percentage']:.2f}%",
            "Distinct Count": f"{col['distinct_count']:,}",
            "Mean Length": f"{col.get('mean_length', 0):.1f}",
        })
df_fields = pd.DataFrame(field_stats)
df_fields

In [ ]:
# 6. Country Distribution (Open-Set Verification)
country_rows = []
for f in profile["files"]:
    if f.get("split") != "train" or f.get("role") != "source":
        continue
    for col in f.get("columns", []):
        if isinstance(col, dict) and "country_distribution" in col:
            for country, count in col["country_distribution"].items():
                country_rows.append({
                    "File": f["filename"],
                    "Country": country,
                    "Count": f"{count:,}",
                    "Percentage": f"{count / f['row_count'] * 100:.2f}%"
                })
df_country = pd.DataFrame(country_rows)
df_country

In [ ]:
# 7. Real Observed Noise Patterns & Entity Variation Examples
print("=== Real Observed Variation Examples Between Linked Entities ===")
examples_dict = profile.get("variations", {}).get("examples", {})
for cat_name, examples in list(examples_dict.items())[:8]:
    print(f"\nCategory: {cat_name}")
    for ex in examples[:2]:
        s1_val = str(ex.get('left_value', '')).encode('ascii', errors='replace').decode('ascii')
        match_val = str(ex.get('right_value', '')).encode('ascii', errors='replace').decode('ascii')
        print(f"  S1 ({ex.get('left_id', '')}): {s1_val}")
        print(f"  Match ({ex.get('right_id', '')}): {match_val}")
        print(f"  Detail: {ex.get('detail', '')}")

In [ ]:
# 8. Leakage Verification and Validation Strategy
leakage = profile.get("leakage", {})
print("=== Leakage Verification ===")
for k, v in leakage.items():
    print(f"{k}: {v}")

strategy = profile.get("validation_strategy", {})
print("\n=== Recommended Validation Strategy for Phase 3 ===")
for rec in strategy.get("recommendations", []):
    print(f"- {rec}")